# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes the accepted jobs to `jobs/2-llm_filtered_jobs.jsonl`.


In [6]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

### Check and load scraped jobs from file

In [5]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

if jobs_df.empty:
    raise ValueError("The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first.")

required_columns = ["title", "job_url", "description"]

for column in required_columns:
    if column not in jobs_df.columns:
        raise KeyError(f"The scraped jobs JSONL file must contain a '{column}' column.")

    missing_count = int(jobs_df[column].isna().sum())
    if missing_count > 0:
        raise ValueError(f"The scraped jobs JSONL file contains {missing_count} missing values in '{column}'.")

    empty_count = int((jobs_df[column].astype(str).str.strip() == "").sum())
    if empty_count > 0:
        raise ValueError(f"The scraped jobs JSONL file contains {empty_count} empty values in '{column}'.")

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 15 entries!


### Use the LLM as a classifier

We now ask an LLM to decide whether each job is really an AI engineering role based on the job description.

In [ ]:
filtered_jobs_path = Path("jobs") / "2-llm_filtered_jobs.jsonl"

jobs_df = jobs_df.copy()

for column in required_columns:
    jobs_df[column] = jobs_df[column].astype(str).str.strip()

jobs_df = jobs_df.reset_index(drop=True)

client = OpenAI()

screening_instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- An AI Engineer is not mainly a model researcher and does not primarily build models from scratch.
- AI engineering is closer to product and application engineering than to core ML research.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

screening_schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

max_description_chars = 10000
screening_results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    company = job["company"]
    job_url = job["job_url"]
    description = job["description"]

    description_for_prompt = description[:max_description_chars]
    if len(description) > max_description_chars:
        description_for_prompt += "\n\n[Description truncated for screening.]"

    print(f"Screening job {i}/{len(jobs_df)}: {title} @ {company}")

    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=screening_instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\nCompany: {company}\nURL: {job_url}\n\nDescription:\n{description_for_prompt}""",
        max_output_tokens=120,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": screening_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = result["is_ai_engineering_role"]
    reason = result["reason"].strip()

    screening_results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

screening_results_df = pd.DataFrame(screening_results)
screened_jobs = pd.concat([jobs_df, screening_results_df], axis=1)

ai_engineer_jobs = screened_jobs[screened_jobs["is_ai_engineering_role"]].copy()
non_ai_engineer_jobs = screened_jobs[~screened_jobs["is_ai_engineering_role"]].copy()
ai_engineer_jobs.to_json(filtered_jobs_path, orient='records', lines=True, force_ascii=False)

print(f"Saved LLM-filtered jobs to: {filtered_jobs_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {len(ai_engineer_jobs)}")
print(f"Jobs classified as not AI engineering or unclear: {len(non_ai_engineer_jobs)}")

screening_results_to_show = screened_jobs[["title", "company", "is_ai_engineering_role", "llm_reason", "job_url"]].copy()
screening_results_to_show["job_url"] = screening_results_to_show["job_url"].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) and str(url).strip() else ''
)

display(HTML(screening_results_to_show.to_html(escape=False, index=False)))
